In [1]:
import os
import shutil
import pandas as pd
import cv2
import numpy as np

from sklearn.model_selection import train_test_split

In [2]:
# Raw data paths
RAW_DIR = r"E:/ML Portfolio Projects/11. Retinopathy_Detection/data/raw"
IMAGES_DIR = os.path.join(RAW_DIR, "images")
CSV_PATH = os.path.join(RAW_DIR, "data_all.csv")

# Processed data path
PROCESSED_DIR = r"E:/ML Portfolio Projects/11. Retinopathy_Detection/data/processed"

In [3]:
df = pd.read_csv(CSV_PATH)

# Keep only required columns
df = df[['file', 'cat']]

print("Total Images:", len(df))
print("\nClass Distribution:")
print(df['cat'].value_counts().sort_index())

Total Images: 1764

Class Distribution:
cat
1    811
2    569
3    384
Name: count, dtype: int64


**STRATIFIED SPLIT (70% TRAIN / 15% VAL / 15% TEST)**

In [4]:
# First split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['cat'],
    random_state=42
)

# Second split: 15% val, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['cat'],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 1234
Validation: 265
Test: 265


In [5]:
print("Train Class Distribution:")
print(train_df['cat'].value_counts().sort_index())

print("\nValidation Class Distribution:")
print(val_df['cat'].value_counts().sort_index())

print("\nTest Class Distribution:")
print(test_df['cat'].value_counts().sort_index())

Train Class Distribution:
cat
1    567
2    398
3    269
Name: count, dtype: int64

Validation Class Distribution:
cat
1    122
2     85
3     58
Name: count, dtype: int64

Test Class Distribution:
cat
1    122
2     86
3     57
Name: count, dtype: int64


In [6]:
def apply_clahe(image_path, target_size=(224, 224)):
    # Read image
    img = cv2.imread(image_path)

    # Convert BGR to LAB
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    # Split LAB channels
    l, a, b = cv2.split(lab)

    # Apply CLAHE to L channel
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    cl = clahe.apply(l)

    # Merge channels
    merged_lab = cv2.merge((cl, a, b))

    # Convert LAB back to BGR
    enhanced_img = cv2.cvtColor(merged_lab, cv2.COLOR_LAB2BGR)

    # Resize
    enhanced_img = cv2.resize(enhanced_img, target_size)

    return enhanced_img

In [7]:
## Without CLAHE

# def copy_images(dataframe, split_name):
#     copied_count = 0
#     missing_count = 0

#     for _, row in dataframe.iterrows():
#         file_name = row['file']
#         class_label = f"class_{row['cat']}"

#         src_path = os.path.join(IMAGES_DIR, file_name)
#         dst_folder = os.path.join(PROCESSED_DIR, split_name, class_label)
#         dst_path = os.path.join(dst_folder, file_name)

#         # Ensure destination folder exists
#         os.makedirs(dst_folder, exist_ok=True)

#         # Copy image
#         if os.path.exists(src_path):
#             shutil.copy2(src_path, dst_path)
#             copied_count += 1
#         else:
#             missing_count += 1
#             print(f"Missing file: {src_path}")

#     print(f"{split_name.upper()} → Copied: {copied_count}, Missing: {missing_count}")

In [8]:
def copy_images(dataframe, split_name):
    copied_count = 0
    missing_count = 0

    for _, row in dataframe.iterrows():
        file_name = row['file']
        class_label = f"class_{row['cat']}"

        src_path = os.path.join(IMAGES_DIR, file_name)

        dst_folder = os.path.join(PROCESSED_DIR, split_name, class_label)

        dst_path = os.path.join(dst_folder, file_name)

        # Ensure destination folder exists
        os.makedirs(dst_folder, exist_ok=True)

        # Process and save image
        if os.path.exists(src_path):

            # Apply CLAHE + resize
            processed_img = apply_clahe(src_path)

            # Save processed image
            cv2.imwrite(dst_path, processed_img)

            copied_count += 1

        else:
            missing_count += 1
            print(f"Missing file: {src_path}")

    print(f"{split_name.upper()} → Saved: {copied_count}, Missing: {missing_count}")

In [9]:
copy_images(train_df, "train")
copy_images(val_df, "val")
copy_images(test_df, "test")

print("Dataset split and copy completed successfully.")

TRAIN → Saved: 1234, Missing: 0
VAL → Saved: 265, Missing: 0
TEST → Saved: 265, Missing: 0
Dataset split and copy completed successfully.
